<h2 align="center" style="color:purple">Bank Credit Card Project</h2>

### **Objective:** 

Analyze customers' transactions and credit profiles to figure out a target group for the launch of AtliQo bank credit card

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")



<h1 style="color:purple" align="center">Data Import<h1>

### **Load the Data with CSV file**

In [ ]:
df_cust = pd.read_csv('datasets/customers.csv')
df_cs = pd.read_csv('datasets/credit_profiles.csv')
df_trans = pd.read_csv('datasets/transactions.csv')

 **Read it from MySQL for Loading Data to showcase how im connecting Sql into Python**

In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host='localhost',
    user='root',
    passwd='sree0509',
    database='e_master_card'
)

df_cust_sql = pd.read_sql("SELECT * FROM customers", conn)
df_cust.head(3)

In [ ]:
df_trans_sql = pd.read_sql("SELECT * FROM transactions", conn)
df_trans.head()

In [ ]:
df_cs_sql = pd.read_sql("SELECT * FROM credit_profiles", conn)
df_cs.head(3)

**when you are done importing the data, close the connection**

In [ ]:

conn.close()

In [ ]:
print("Customers data",df_cust.shape)
print("Credit Score data",df_cs.shape)
print("Transactions data",df_trans.shape)

In [ ]:
df_cust.head()

In [ ]:
df_cs.head()

In [ ]:
df_trans.head()

<h1 style="color:purple" align="center">Explore Customers Table<h1>

In [ ]:
df_cust.head(3)

In [ ]:
df_cust.describe()

<h2 style="color:Blue">1. Analyze Income Column<h2>

### Handle Null Values: Annual income

"I checked for null values in the DataFrame using the .isnull().sum() method. This method helped me identify the number of missing values in each column, allowing me to determine which columns needed data cleaning."

In [ ]:
df_cust.isnull().sum()

In [ ]:
df_cust.isna().sum()

I handled these null values in different ways, depending on the data requirements

1. **Remove them**: Since there are 50 of them in a dataframe of 1000, we will not remove them as we don't want to loose some important records
1. **Replace them with mean or median**: It is suggested with use median in the case of income. This is because in an income data there could be outliers and median is more robust to these outliers
1. **Replace them with median per occupation**: Occupation wise median income can vary. It is best to use a median per occupation for replacement

In [ ]:
occupation_wise_inc_median = df_cust.groupby("occupation")["annual_income"].median()
occupation_wise_inc_median

In [ ]:
occupation_wise_inc_median['Artist']

In [ ]:
# 2. Replace null values in annual_income with the median income of their occupation group
df_cust['annual_income'] = df_cust.apply(
    lambda row: occupation_wise_inc_median[row['occupation']] if pd.isnull(row['annual_income']) else row['annual_income'],
    axis=1
)

In [ ]:
df_cust.iloc[[1,29]]

**Previously, records at location 1 and 29 had null annual income. Now, a median value per occupation has been assigned to fill those missing values.**

In [ ]:
df_cust.isnull().sum()

 Number of null values in all the columns is zero now

Now that there are no null values, let us view the distribution of annual income

plt.figure(figsize=(5, 5))
sns.histplot(df_cust['annual_income'], kde=True, color='green', label='Data')
plt.title('Histogram of annual_income')
plt.show()

**The income distribution was right-skewed, as observed earlier. Now, the describe() function was used to check some quick statistics.**

In [ ]:
df_cust.describe()

**The following observations were made from the above:**

Age: Minimum = 1, Maximum = 135

Annual Income: Minimum = 2, Maximum = 447K

The age column contained outliers. Annual income also seemed to have outliers, especially in the minimum values, since the business suggested that the minimum income should be at least 100.

In [ ]:
df_cust.annual_income.describe()

### Outlier Detection: Annual income

Standard deviation was used to detect outliers. A common practice is to treat any value beyond ±3 standard deviations as an outlier.


In [ ]:
df_cust['annual_income'].mean(), df_cust['annual_income'].std()

In [ ]:
lower = df_cust['annual_income'].mean() - 3*df_cust['annual_income'].std()
upper = df_cust['annual_income'].mean() + 3*df_cust['annual_income'].std()

lower, upper

In [ ]:
df_cust[df_cust['annual_income']>upper]

Two outliers were detected based on the statistical criteria of ±3 standard deviations. However, not all such values are always considered outliers. Business knowledge and judgment must be applied.

After discussing with the business, it was concluded that higher incomes for business owners are usual, so these data points were retained to maintain realistic analysis.

On the lower end, the minimum income was 2. The business manager stated that the income should be at least 100. This criterion was used to identify outliers on the lower end, as such values might have resulted from a data error

In [ ]:
df_cust[df_cust.annual_income<100]

Records with income less than $100 were identified as outliers. 

**The following options were considered for treatment:**

**1) Remove them:** After discussions with the business, removal was not chosen since these are valid customers and should be included in the analysis.

**2) Replace them with the mean or median:** The mean is sensitive to outliers, so the median is a better choice for income values.

**3) Replace them with the occupation-wise median:** Income levels vary by occupation. For example, the median income for a data scientist differs from that of a business owner. Replacing with the occupation-wise median was considered the best approach.

In [ ]:
occupation_wise_inc_median["Artist"]

In [ ]:
for index, row in df_cust.iterrows():
    if row["annual_income"] < 100:
        occupation = df_cust.at[index, "occupation"]
        df_cust.at[index, "annual_income"] = occupation_wise_inc_median[occupation]

In [ ]:
df_cust[df_cust.annual_income<100]

In [ ]:
df_cust.loc[[112,256]]

Records at locations 112 and 256 had an annual income of less than $100. Now, these values have been replaced with the median income per occupation

### Data Visualization: Annual Income

let explore average income level based on occupation, gender, location and marital status

In [ ]:
avg_income_per_occupation  = df_cust.groupby("occupation")["annual_income"].mean()
avg_income_per_occupation 

In [ ]:
# List of categorical columns
cat_cols = ['gender', 'location', 'occupation', 'marital_status']

num_rows = 3
# Create subplots
fig, axes = plt.subplots(num_rows, 2, figsize=(12, 4 * num_rows))

# Flatten the axes array to make it easier to iterate
axes = axes.flatten()

# Create subplots for each categorical column
for i, cat_col in enumerate(cat_cols):
    # Calculate the average annual income for each category
    avg_income_by_category = df_cust.groupby(cat_col)['annual_income'].mean().reset_index()
    
    # Sort the data by 'annual_income' before plotting
    sorted_data = avg_income_by_category.sort_values(by='annual_income', ascending=False)
    
    sns.barplot(x=cat_col, y='annual_income', data=sorted_data, ci=None, ax=axes[i], palette='tab10')
    axes[i].set_title(f'Average Annual Income by {cat_col}')
    axes[i].set_xlabel(cat_col)
    axes[i].set_ylabel('Average Annual Income')

    # Rotate x-axis labels for better readability
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45)

# Hide any unused subplots
for i in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[i])
plt.tight_layout()
plt.show()

<h2 style="color:Blue">2. Analyze Age Column<h2>

### Handle Null Values: Age Column

First, the Age column was checked for any NULL values

In [ ]:
df_cust.age.isnull().sum()

No NULL values were found in the Age column. This means handling missing values is not necessary.

In [ ]:
df_cust.describe()

### Outlier Treatment: Age

The minimum age was 1, and the maximum age was 135, which seemed to be outliers. To investigate further, the age distribution was analyzed

In [ ]:
min_age = df_cust.age.min()
max_age = df_cust.age.max()

min_age, max_age

In [ ]:
plt.hist(df_cust.age, bins=20, edgecolor='black')
plt.xlabel("Age")
plt.ylabel("Count")
plt.title("Customer Age Distribution")

plt.axvline(min_age, color="red", label=f"Min Age: {min_age}")
plt.axvline(max_age, color="green", label=f"Max Age: {max_age}")

plt.legend()
plt.show()

From the age distribution, customers above 80 and below 15 were identified as potential outliers.

In [ ]:
df_cust[(df_cust.age<15)|(df_cust.age>80)]

In [ ]:
outliers = df_cust[(df_cust.age<15)|(df_cust.age>80)]
outliers.shape

A total of 20 outliers were found in the Age column. The following options were considered for handling them:

**1) Remove them:** This was not preferred, as it would result in losing important information.

**2) Replace outlier values with an appropriate value:** The mean or median could be used for replacement, with the median being a better choice since it is less affected by extreme values.

In [ ]:
df_cust.age.median()

**Instead of replacing outliers with the overall median age, a better approach is to calculate and use the median age per occupation. This ensures that age values remain realistic within each profession**

In [ ]:
outliers.head(3)

As observed, the median age for business owners is 49, while artists have the youngest median age

**The median age per occupation was calculated and used to replace outliers. This ensures that age values remain realistic within each profession.**

In [ ]:
median_age_per_occupation = df_cust.groupby('occupation')['age'].median()
median_age_per_occupation

In [ ]:
for index, row in outliers.iterrows():
    if pd.notnull(row['age']):
        occupation = df_cust.at[index, 'occupation']
        df_cust.at[index, 'age'] = median_age_per_occupation[occupation]

In [ ]:
df_cust[(df_cust.age<15)|(df_cust.age>80)]

In [ ]:
df_cust.age.describe()

**As seen above, all outliers have been handled. The minimum age is now 18, and the maximum age is 64**

In [ ]:
df_cust.head()

### Data Visualization: Age Column

In [ ]:
# Define the bin edges and labels
bin_edges = [17, 25, 48, 65]  # Adjust as needed
bin_labels = ['18-25', '26-48', '49-65']

# Use the cut function to bin and label the age column
pd.cut(df_cust['age'], bins=bin_edges, labels=bin_labels)

In [ ]:
# Define the bin edges and labels
bin_edges = [17, 25, 48, 65]  # Adjust as needed
bin_labels = ['18-25', '26-48', '49-65']

# Use the cut function to bin and label the age column
df_cust['age_group'] = pd.cut(df_cust['age'], bins=bin_edges, labels=bin_labels)

In [ ]:
df_cust.head()

In [ ]:
df_cust['age_group'].value_counts(normalize=True)*100

In [ ]:
# Calculate the count of values in each age group
age_group_counts = df_cust['age_group'].value_counts(normalize=True) * 100

# Plot the pie chart
plt.figure(figsize=(4, 4))
plt.pie(
    age_group_counts, 
    labels=age_group_counts.index, 
    explode=(0.1,0,0), 
    autopct='%1.1f%%', 
    shadow=True,
    startangle=140)
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.title('Distribution of Age Groups')
plt.show()

##### More than 50% of customer base are in in age group of 26 - 48 adn ~26% are of age group 18 - 25

<h2 style="color:Blue">3. Analyze Gender and Location Distribution<h2>

In [ ]:
customer_location_gender = df_cust.groupby(['location', 'gender']).size().unstack(fill_value=0)

# Create a stacked bar chart to visualize the distribution of payment types for each occupation
customer_location_gender.plot(kind='bar', stacked=True, figsize=(5, 4))

# Add labels and title
plt.xlabel('Location')
plt.ylabel('Count')
plt.title('Customer Distribution by Location and Gender')

# Show the bar chart
plt.legend(title='Payment Type', bbox_to_anchor=(1, 1))  # Add a legend

# Rotate the x-axis labels for better readability
plt.xticks(rotation=45)

plt.show()

<h1 style="color:purple" align="center">Explore Credit Score Table<h1>

In [ ]:
df_cs.head()

### Data Cleaning Step 1: Remove Duplicates 

In [ ]:
df_cs.shape

**There are 1004 rows in this dataframe whereas customers dataframe had only 1000. There might be invalid or duplicate data in df_cs**

In [ ]:
df_cs['cust_id'].nunique()

In [ ]:
df_cs.duplicated('cust_id')

In [ ]:
df_cs[df_cs.duplicated('cust_id', keep=False)]

In [ ]:
df_cs_clean_1 = df_cs.drop_duplicates(subset='cust_id', keep="last")
df_cs_clean_1.shape

In [ ]:
df_cs_clean_1[df_cs_clean_1.duplicated('cust_id', keep=False)]

df_cs_clean_1 looks clean now after cleaning duplicates.

Next step would be to see if there are any null values

### Data Cleaning Step 2: Handle Null Values

In [ ]:
df_cs_clean_1.isnull().sum()

**A significant number of NULL values were found in the credit_limit column. Based on business knowledge, the credit limit depends on the credit score of a customer. The goal is to explore whether a mathematical relationship exists between the credit score and credit limit, which can then be used to fill in missing values**

In [ ]:
df_cs_clean_1[df_cs_clean_1.credit_limit.isnull()]

In [ ]:
df_cs_clean_1['credit_limit'].unique()

**The credit_limit column has only a few unique values. Checking the count of each unique value helps understand its distribution.**

In [ ]:
df_cs_clean_1['credit_limit'].value_counts()

In [ ]:
# Looking at scatter plot for credit score vs credit_limit again (after handling oultiers)
# Create a scatter plot
plt.figure(figsize=(20, 5))
plt.scatter(df_cs_clean_1['credit_limit'], df_cs_clean_1['credit_score'], c='blue', marker='o', label='Data Points')

# Customize the plot
plt.title('Credit Score vs. Credit Limit')
plt.xlabel('Credit Limit')
plt.ylabel('Credit Score')

# Adjust the y-axis bin interval to 1000
plt.xticks(range(0, 90001, 5000))
plt.grid(True)

# Show the plot
plt.legend()
plt.show()

**A clear relationship was observed between credit score and credit limit. The credit limit follows a tiered structure:**

Credit Score ≤ 650 → Minor credit limit (< $1,000)

Credit Score 650–700 → Around $20,000

Credit Score 700–750 → Around $40,000

Higher scores → Even higher limit

In [ ]:
# Define bin ranges
bin_ranges = [300, 450, 500, 550, 600, 650, 700, 750, 800]

# Create labels for the bins
bin_labels = [f'{start}-{end-1}' for start, end in zip(bin_ranges, bin_ranges[1:])]

# Use pd.cut to assign data to bins
df_cs_clean_1['credit_score_range'] = pd.cut(df_cs_clean_1['credit_score'], bins=bin_ranges, labels=bin_labels, include_lowest=True, right=False)

In [ ]:
df_cs_clean_1.head()

**A new column, credit_score_range, was created based on the credit_score column.**

In [ ]:
df_cs_clean_1[['credit_score','credit_score_range', 'credit_limit']].head(3)

In [ ]:
df_cs_clean_1[df_cs_clean_1['credit_score_range']=="750-799"]

In [ ]:
df_cs_clean_1[df_cs_clean_1['credit_score_range']=="300-449"]

For each credit_score_range, the most frequently occurring credit_limit was identified using the MODE function. This helps determine the most common credit limit assigned to each credit score range

In [ ]:
mode_df = df_cs_clean_1.groupby('credit_score_range')['credit_limit'].agg(lambda x: x.mode().iloc[0]).reset_index()
mode_df

In [ ]:
df_cs_clean_1[df_cs_clean_1.credit_limit.isnull()].sample(3)

In [ ]:
# Merge the mode values back with the original DataFrame
df_cs_clean_2 = pd.merge(df_cs_clean_1, mode_df, on='credit_score_range', suffixes=('', '_mode'))
df_cs_clean_2.sample(3)

In [ ]:
df_cs_clean_2[df_cs_clean_2.credit_limit.isnull()].sample(3)

**A new copy of the dataframe was created to ensure reproducibility, keeping the original data intact. Missing values in the credit_limit column were replaced with the most frequently occurring credit_limit for each credit_score_range**

In [ ]:
df_cs_clean_3 = df_cs_clean_2.copy()
df_cs_clean_3['credit_limit'].fillna(df_cs_clean_3['credit_limit_mode'], inplace=True)
df_cs_clean_3.shape

In [ ]:
df_cs_clean_3.isnull().sum()

 **There are zero outliers in the credit_limit column, meaning all NULL values have been successfully handled. The cleaned dataset ensures completeness while maintaining the logical structure of credit limits based on credit scores**

In [ ]:
df_cs_clean_3[df_cs_clean_3.cust_id==117]

Previously customer id 5 had null value in credit_limit. Now it has a valid value

### Data Cleaning Step 3: Handle Outliers: outstanding_debt

In [ ]:
df_cs_clean_3.describe()

Observing the minimum and maximum values of various columns revealed that the maximum outstanding_debt exceeds the maximum credit_limit. However, based on business understanding, a customer cannot have debt exceeding their credit limit. The next step is to identify how many such cases exist in the dataset

**Visualizing outliers**

In [ ]:
plt.figure(figsize=(5, 5))
sns.boxplot(x=df_cs_clean_3['outstanding_debt'])
plt.title('Box plot for outstanding debt')

Instead of using statistical approaches like standard deviation or IQR, business knowledge is applied to identify outliers. Any outstanding_debt that exceeds the credit_limit is considered an outlier, as customers are not allowed to exceed their credit limit.

In [ ]:
df_cs_clean_3[df_cs_clean_3.outstanding_debt>df_cs_clean_3.credit_limit]

Outliers in outstanding_debt (where the value exceeds credit_limit) will be replaced with credit_limit. This assumes a data processing error caused these high values, and adjusting them ensures consistency with business rules.

In [ ]:
df_cs_clean_3.loc[df_cs_clean_3['outstanding_debt'] > df_cs_clean_3['credit_limit'], 'outstanding_debt']

In [ ]:
df_cs_clean_3.loc[df_cs_clean_3['outstanding_debt'] > df_cs_clean_3['credit_limit'], 'outstanding_debt'] = df_cs_clean_3['credit_limit']

In [ ]:
df_cs_clean_3.loc[[55,66]]

In [ ]:
df_cs_clean_3[df_cs_clean_3.outstanding_debt>df_cs_clean_3.credit_limit]

All outliers in column outstanding_debt are now GONE. 

In [ ]:
df_cs_clean_3.describe()

In [ ]:
for col in df_cs_clean_3:
    sns.boxplot(x=df_cs_clean_3[col])
    plt.show()

### Data Exploration: Visualizing Correlation in Credit Score Table

In [ ]:
df_cust.head(2)

In [ ]:
df_cs_clean_3.head(2)

In [ ]:
df_merged = df_cust.merge(df_cs_clean_3, on='cust_id', how='inner')
df_merged.head(2)

In [ ]:
numerical_cols = ['credit_score', 'credit_utilisation', 'outstanding_debt', 'credit_limit', 'annual_income','age']

correlation_matrix = df_merged[numerical_cols].corr()
correlation_matrix

In [ ]:
# Create a heatmap of the correlation matrix
plt.figure(figsize=(5, 3))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.8)
plt.title('Correlation Plot')
plt.savefig("Correlation.png", dpi=400, bbox_inches="tight", pad_inches=0.1)
plt.show()
plt.show()

A high correlation (~0.85) is observed between credit limit and credit score, indicating a strong relationship. Additionally, credit limit and annual income also show high correlation.

This correlation table helps in understanding relationships between variables, which can be useful for further analysis and feature selection in modeling.

In [ ]:
# Just looking if there is any relation between annual_income and credit score
plt.figure(figsize=(5, 4))
sns.scatterplot(x='annual_income', y='credit_score', data=df_merged, alpha=0.5)
plt.title('Scatter Plot of Annual income vs credit score')
plt.xlabel('Annual Income')
plt.ylabel('Credit Score')
plt.show()

##### No clear pattern observed

<h1 style="color:purple" align="center">Transactions Table<h1>

In [ ]:
df_trans.head(2)

In [ ]:
df_trans.shape

### Data Cleaning Step 1: Handle NULL Values: platform column

In [ ]:
df_trans.isnull().sum()

platform has a lot of null values. Let's check them further

In [ ]:
df_trans[df_trans.platform.isnull()]

In [ ]:
sns.countplot(y='product_category', hue='platform', data=df_trans)

In the above chart, Amazon is the most frequently used platform across all product categories. Since it dominates in usage, missing values in the platform column can be replaced with 'Amazon', assuming that most users likely purchased from there

In [ ]:
df_trans.platform.mode()

In [ ]:
df_trans.platform.mode()[0]

In [ ]:
df_trans['platform'].fillna(df_trans.platform.mode()[0], inplace=True)

In [ ]:
df_trans.isnull().sum()

Once again, all NULL values have been successfully handled. The platform column now has no missing values, ensuring completeness in the dataset

### Data Cleaning Step 2: Treat Outliers: tran_amount

In [ ]:
df_trans.describe()

We can see transactions with 0 amount. These seem to be invalid

In [ ]:
df_trans_zero = df_trans[df_trans.tran_amount==0]
df_trans_zero.head(3)

In [ ]:
df_trans_zero.shape

In [ ]:
df_trans_zero.platform.value_counts()

In [ ]:
df_trans_zero[['platform','product_category','payment_type']].value_counts()

**It appears that zero transaction values occur when:**

**Platform** = Amazon

**Product_Category** = Electronics

**Payment_Type** = Credit Card

To correct this, other transactions within the same group will be identified, and the median transaction value will be used for replacement. The median is chosen over the mean to avoid the impact of outliers.

In [ ]:
df_trans_1 = df_trans[(df_trans.platform=='Amazon')&(df_trans.product_category=="Electronics")&(df_trans.payment_type=="Credit Card")]
df_trans_1.shape

In [ ]:
df_trans_1[df_trans_1.tran_amount>0]

In [ ]:
median_to_replace = df_trans_1[df_trans_1.tran_amount>0].tran_amount.median()
median_to_replace

In [ ]:
df_trans['tran_amount'].replace(0,median_to_replace, inplace=True)

In [ ]:
df_trans[df_trans.tran_amount==0]

As above, there were no zero values are left in tran_amount column

In [ ]:
df_trans.tran_amount.describe()

In [ ]:
df_trans[df_trans['tran_amount']<1000].describe()

In [ ]:
Q1, Q3 = df_trans['tran_amount'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower = Q1 - 2 * IQR
upper = Q3 + 2 * IQR

lower, upper

In [ ]:
df_trans[df_trans.tran_amount<upper].tran_amount.max()

In [ ]:
df_trans[df_trans.tran_amount>upper].tran_amount.min()

In [ ]:
df_trans_outliers = df_trans[df_trans.tran_amount>=upper]
df_trans_outliers

In [ ]:
df_trans_normal = df_trans[df_trans.tran_amount<upper]
df_trans_normal

In [ ]:
tran_mean_per_category = df_trans_normal.groupby("product_category")["tran_amount"].mean()
tran_mean_per_category

In [ ]:
df_trans.loc[df_trans_outliers.index]

In [ ]:
df_trans.loc[df_trans_outliers.index, 'tran_amount'] = df_trans_outliers['product_category'].map(tran_mean_per_category)

In [ ]:
df_trans.loc[df_trans_outliers.index]

You can now see that we got rid of outliers from tran_amount column. Great job folks 👍🏼🔥🔥

In [ ]:
df_trans.describe()

In [ ]:
sns.histplot(x='tran_amount', data=df_trans, bins=20, kde=True)

Above shows the histogram of transactions after the removal of outliers. You can see that distribution is right skewed. Transaction amount now is less than 1000

### Data Visualization: Payment Type Distribution

In [ ]:
df_trans.head(3)

In [ ]:
sns.countplot(x=df_trans.payment_type, stat='percent')

**Distribution of payment types across age groups**

In [ ]:
df_merged_2 = df_merged.merge(df_trans, on='cust_id', how='inner')
df_merged_2.head(3)

In [ ]:
df_merged_2.shape

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x='age_group', hue='payment_type', data=df_merged_2, palette='Set3')
plt.title('Distribution of Payment types across Age groups')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Payment Type', loc='upper right')

plt.show()

From above analysis, The age group 18-25 has less exposure to credit cards compared to other groups

In [ ]:
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(12,5))

sns.countplot(x='age_group', hue="product_category", data=df_merged_2, ax=ax1)
ax1.set_title("Product Category Count By Age Group")
ax1.set_xlabel("Age Group")
ax1.set_ylabel("Count")
ax1.legend(title="Product Category", loc='upper right')

sns.countplot(x='age_group', hue="platform", data=df_merged_2, ax=ax2)
ax2.set_title("Platform Count By Age Group")
ax2.set_xlabel("Age Group")
ax2.set_ylabel("Count")
ax2.legend(title="Product Category", loc='upper right')

plt.show()

### Observations:

1. Top 3 purchasing categories of customers in age group (18 -25) : Electronics, Fashion & Apparel, Beauty & personal care
1. Top platforms : Amazon, Flipkart, Alibaba

### Data Visualization: Average Transaction Amount

In [ ]:
# List of categorical columns
cat_cols = ['payment_type', 'platform', 'product_category', 'marital_status', 'age_group']

num_rows = 3
# Create subplots
fig, axes = plt.subplots(num_rows, 2, figsize=(12, 4 * num_rows))

# Flatten the axes array to make it easier to iterate
axes = axes.flatten()

# Create subplots for each categorical column
for i, cat_col in enumerate(cat_cols):
    # Calculate the average annual income for each category
    avg_tran_amount_by_category = df_merged_2.groupby(cat_col)['tran_amount'].mean().reset_index()
    
    # Sort the data by 'annual_income' before plotting
    sorted_data = avg_tran_amount_by_category.sort_values(by='tran_amount', ascending=False)
    
    sns.barplot(x=cat_col, y='tran_amount', data=sorted_data, ci=None, ax=axes[i], palette='tab10')
    axes[i].set_title(f'Average transaction amount by {cat_col}')
    axes[i].set_xlabel(cat_col)
    axes[i].set_ylabel('Average transaction amount')

    # Rotate x-axis labels for better readability
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45)

# Hide any unused subplots
for i in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[i])
plt.tight_layout()
plt.show()

In [ ]:
df_trans.describe()

### Further Analysis On Age Group

Further analysis on age group to figure out their average income, credit limit, credit score etc

In [ ]:
# Group the data by age group and calculate the average credit_limit and credit_score
age_group_metrics = df_merged.groupby('age_group')[['annual_income', 'credit_limit', 'credit_score']].mean().reset_index()
age_group_metrics

In [ ]:
# Create subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))

# Plot 1: Average annual income by age group
sns.barplot(x='age_group', y='annual_income', data=age_group_metrics, palette='tab10', ax=ax1)
ax1.set_title('Average Annual Income by Age Group')
ax1.set_xlabel('Age Group')
ax1.set_ylabel('Average Annual Income')
ax1.tick_params(axis='x', rotation=0)

# Plot 2: Average Max Credit Limit by Age Group
sns.barplot(x='age_group', y='credit_limit', data=age_group_metrics, palette='hls', ax=ax2)
ax2.set_title('Average Credit Limit by Age Group')
ax2.set_xlabel('Age Group')
ax2.set_ylabel('Average Credit Limit')
ax2.tick_params(axis='x', rotation=0)

# Plot 3: Average Credit Score by Age Group
sns.barplot(x='age_group', y='credit_score', data=age_group_metrics, palette='viridis', ax=ax3)
ax3.set_title('Average Credit Score by Age Group')
ax3.set_xlabel('Age Group')
ax3.set_ylabel('Average Credit Score')
ax3.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

<h2 align="center", style="color:purple">Finalize Target Market For a Trial Credit Card Launch<h2>

1. People with age group of 18 -25 accounts to ~26% of customer base in the data
2. Avg annual income of this group is less than 50k
3. They don't have much credit history which is getting reflected in their credit score and credit limit 
4. Usage of credit cards as payment type is relatively low compared to other groups
5. Top 3 most shopping products categories : Electronics, Fashion & Apparel, Beauty & Personal care

<h2 align="center", style="color:red"> Data Visualization<h2>

In [ ]:
# Create 3x3 grid layout
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

# === Pie Chart (First Plot) ===
age_group_counts = df_cust['age_group'].value_counts(normalize=True) * 100
colors = sns.color_palette("pastel")  # Soft color tones
axes[0].pie(age_group_counts, labels=age_group_counts.index, explode=(0.1, 0, 0), 
            autopct='%1.1f%%', shadow=True, startangle=140, colors=colors, textprops={'fontsize': 12})
axes[0].set_title('Distribution of Age Groups', fontsize=14)

# === Categorical Count Plots ===
plots = [
    ('payment_type', 'Payment Types', 'Set3'),
    ('product_category', 'Product Category', 'coolwarm'),
    ('platform', 'Platform', 'husl')
]
#For loop  Function
for i, (col, title, palette) in enumerate(plots, start=1):
    sns.countplot(x='age_group', hue=col, data=df_merged_2, ax=axes[i], palette=palette)
    axes[i].set_title(f'{title} Count by Age Group', fontsize=14)
    axes[i].set_xlabel("Age Group", fontsize=12)
    axes[i].set_ylabel("Count", fontsize=12)
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(title=title, fontsize=8, title_fontsize=12, loc='upper right', frameon=True)

# === Avg Transaction Amount by Category ===
cat_cols = [
    ('payment_type', 'tab10'),
    ('platform', 'viridis'),
    ('product_category', 'rocket'),
    ('marital_status', 'magma'),
    ('age_group', 'coolwarm')
]

for i, (col, palette) in enumerate(cat_cols, start=4):
    avg_tran = df_merged_2.groupby(col)['tran_amount'].mean().reset_index().sort_values(by='tran_amount', ascending=False)
    sns.barplot(x=col, y='tran_amount', data=avg_tran, ax=axes[i], palette=palette)
    axes[i].set_title(f'Avg Transaction Amount by {col}', fontsize=14)
    axes[i].set_xlabel(col, fontsize=12)
    axes[i].set_ylabel('Avg Transaction Amount', fontsize=12)
    axes[i].tick_params(axis='x', rotation=45)

# === Formatting Adjustments ===
plt.tight_layout()
plt.savefig("Analysis.png", dpi=400, bbox_inches="tight", pad_inches=0.1)
plt.show()
